In [1]:
import sqlite3
import pandas as pd

In [2]:

# Crear base de datos en memoria
conn = sqlite3.connect(":memory:")
#Definimos los DataFrame
df_accesos_fijos = pd.read_csv("../data/processed/accesos_fijos.csv")
df_abonados_moviles = pd.read_csv("../data/processed/abonados_moviles.csv")
df_trafico_movil = pd.read_csv("../data/processed/trafico_movil.csv")
df_lineas_moviles = pd.read_csv("../data/processed/lineas_moviles.csv")
df_ingresos_por_voz = pd.read_csv("../data/processed/ingresos_por_voz.csv")
df_cobertura_movil_por_municipio = pd.read_csv("../data/processed/cobertura_movil_por_municipio.csv")

#### Las columnas de tecnología en la tabla Cobertura movil por municipio deben pasar a tipo Booleano
##### Esto es para que ocupe menos espacio en memoria y sea mejor el procesamiento. 

In [3]:
#Conversión de tipo de dato STR a BOOLEAN
df_cobertura_movil_por_municipio.dtypes
columnas_cobertura = ["2G","3G","HSPA+","4G","LTE","5G"]

for col in columnas_cobertura:
    df_cobertura_movil_por_municipio[col] = df_cobertura_movil_por_municipio[col].map({"S": True, "N":False})

In [4]:
# Cargar los CSVs procesados como tablas SQL
df_lineas_moviles.to_sql("lineas", conn, index=False, if_exists="replace")
df_ingresos_por_voz.to_sql("ingresos", conn, index=False, if_exists="replace")
df_abonados_moviles.to_sql("abonados", conn, index=False, if_exists="replace")
df_trafico_movil.to_sql("trafico", conn, index=False, if_exists="replace")
df_cobertura_movil_por_municipio.to_sql("cobertura", conn, index=False, if_exists="replace")

128022

In [5]:
# Verificar que las tablas quedaron cargadas
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
print(cursor.fetchall())

[('lineas',), ('ingresos',), ('abonados',), ('trafico',), ('cobertura',)]


### Consulta 1 — ¿Qué operador tiene más líneas en servicio en Q4 2025?

In [6]:
query1 = """
SELECT 
    PROVEEDOR,
    SUM("LÍNEAS EN SERVICIO") as TOTAL_LINEAS,
    SUM("LÍNEAS PREPAGO")    as PREPAGO,
    SUM("LÍNEAS POSPAGO")    as POSPAGO
FROM lineas
WHERE AÑO = 2025 AND TRIMESTRE = 4
GROUP BY PROVEEDOR
ORDER BY TOTAL_LINEAS DESC
"""

resultado1 = pd.read_sql(query1, conn)
print(resultado1)

                                           PROVEEDOR  TOTAL_LINEAS   PREPAGO  \
0                COMUNICACION CELULAR S A COMCEL S A      42665092  30915127   
1            COLOMBIA TELECOMUNICACIONES S.A. E.S.P.      23068639  17713766   
2                            COLOMBIA MOVIL  S.A ESP      17512467  12858125   
3                      PARTNERS TELECOM COLOMBIA SAS       6156330   4959224   
4                      VIRGIN MOBILE COLOMBIA S.A.S.       3323908   3323908   
5                 ALMACENES EXITO INVERSIONES S.A.S.       2431134   2431134   
6                         LOV TELECOMUNICACIONES SAS        200738    200738   
7   EMPRESA DE TELECOMUNICACIONES DE BOGOTA S.A. ESP        168077     62004   
8                                  SUMA MOVIL S.A.S.         52355     52355   
9                                       LIWA SAS ESP         12413     12413   
10                             PLINTRON COLOMBIA SAS          7175      7175   

     POSPAGO  
0   11749965  
1    5354

### Consulta 2 — ¿Cómo evolucionó el tráfico de datos por operador?

In [10]:
# Claro
query2_claro = """
SELECT AÑO, TRIMESTRE,
       ROUND(SUM("TRÁFICO (MB)") / 1000000.0, 2) as TRAFICO_TB
FROM trafico
WHERE PROVEEDOR = 'COMUNICACION CELULAR S A COMCEL S A'
GROUP BY AÑO, TRIMESTRE
ORDER BY AÑO, TRIMESTRE
"""
print("CLARO")
print(pd.read_sql(query2_claro, conn).to_string(index=False))

CLARO
 AÑO  TRIMESTRE  TRAFICO_TB
2023          2   260632.80
2023          3   265890.01
2023          4   289945.63
2024          1   316849.98
2024          2   317630.26
2024          3   324371.46
2024          4   330155.14
2025          1   327504.88
2025          2   336340.86
2025          3   350905.40
2025          4   373397.02


In [11]:
# Tigo
query2_tigo = """
SELECT AÑO, TRIMESTRE,
       ROUND(SUM("TRÁFICO (MB)") / 1000000.0, 2) as TRAFICO_TB
FROM trafico
WHERE PROVEEDOR = 'COLOMBIA MOVIL  S.A ESP'
GROUP BY AÑO, TRIMESTRE
ORDER BY AÑO, TRIMESTRE
"""
print("TIGO")
print(pd.read_sql(query2_tigo, conn).to_string(index=False))

TIGO
 AÑO  TRIMESTRE  TRAFICO_TB
2023          2    98496.89
2023          3   113526.99
2023          4   117558.03
2024          1   117586.40
2024          2   118717.59
2024          3   119686.04
2024          4   117533.80
2025          1   113580.14
2025          2   117226.45
2025          3   118253.43
2025          4   116717.80


In [12]:
# Movistar
query2_movistar = """
SELECT AÑO, TRIMESTRE,
       ROUND(SUM("TRÁFICO (MB)") / 1000000.0, 2) as TRAFICO_TB
FROM trafico
WHERE PROVEEDOR = 'COLOMBIA TELECOMUNICACIONES S.A. E.S.P.'
GROUP BY AÑO, TRIMESTRE
ORDER BY AÑO, TRIMESTRE
"""
print("MOVISTAR")
print(pd.read_sql(query2_movistar, conn).to_string(index=False))

MOVISTAR
 AÑO  TRIMESTRE  TRAFICO_TB
2023          2    56263.19
2023          3    64460.68
2023          4    69892.42
2024          1    73222.42
2024          2    77265.30
2024          3    82279.93
2024          4    80378.32
2025          1    84577.93
2025          2    97670.25
2025          3   108253.48
2025          4   117627.24


### Consulta 3 — ¿Qué departamentos lideran en cobertura 5G?

In [13]:
query3 = """
SELECT 
    DEPARTAMENTO,
    COUNT(DISTINCT MUNICIPIO) as TOTAL_MUNICIPIOS,
    COUNT(DISTINCT CASE WHEN "5G" = 1 THEN MUNICIPIO END) as CON_5G,
    ROUND(COUNT(DISTINCT CASE WHEN "5G" = 1 THEN MUNICIPIO END) * 100.0 / 
          COUNT(DISTINCT MUNICIPIO), 1) as PCT_5G
FROM cobertura
WHERE AÑO = 2025 AND TRIMESTRE = 4
GROUP BY DEPARTAMENTO
ORDER BY CON_5G DESC
"""

resultado3 = pd.read_sql(query3, conn)
print(resultado3)

                                         DEPARTAMENTO  TOTAL_MUNICIPIOS  \
0                                           ANTIOQUIA               125   
1                                        CUNDINAMARCA               116   
2                                     VALLE DEL CAUCA                42   
3                                           SANTANDER                87   
4                                                META                29   
5                                  NORTE DE SANTANDER                40   
6                                              BOYACÁ               123   
7                                              TOLIMA                47   
8                                           RISARALDA                14   
9                                             QUINDÍO                12   
10                                             NARIÑO                64   
11                                              HUILA                37   
12                       

### Consulta 4 — ¿Qué operador genera más ingreso por línea?

In [9]:
query4 = """
SELECT 
    l.PROVEEDOR,
    SUM(l."LÍNEAS EN SERVICIO")            as LINEAS,
    SUM(i."INGRESOS OPERACIONALES")        as INGRESOS,
    ROUND(SUM(i."INGRESOS OPERACIONALES") / 
          SUM(l."LÍNEAS EN SERVICIO"), 0)  as INGRESO_POR_LINEA
FROM lineas l
JOIN ingresos i 
    ON l.PROVEEDOR = i.PROVEEDOR
    AND l.AÑO     = i.AÑO
    AND l.TRIMESTRE = i.TRIMESTRE
WHERE l.AÑO = 2025 AND l.TRIMESTRE = 4
GROUP BY l.PROVEEDOR
ORDER BY INGRESO_POR_LINEA DESC
"""

resultado4 = pd.read_sql(query4, conn)
print(resultado4)

                                           PROVEEDOR    LINEAS      INGRESOS  \
0   EMPRESA DE TELECOMUNICACIONES DE BOGOTA S.A. ESP    168077    1594994901   
1            COLOMBIA TELECOMUNICACIONES S.A. E.S.P.  23068639  145877744662   
2                            COLOMBIA MOVIL  S.A ESP  17512467  104022148200   
3                COMUNICACION CELULAR S A COMCEL S A  42665092  161690113479   
4                                       LIWA SAS ESP     12413      24559321   
5                      PARTNERS TELECOM COLOMBIA SAS   6156330   10022725525   
6                 ALMACENES EXITO INVERSIONES S.A.S.   2431134    3453168290   
7                                  SUMA MOVIL S.A.S.     52355      39562469   
8                      VIRGIN MOBILE COLOMBIA S.A.S.   3323908    2130344867   
9                              PLINTRON COLOMBIA SAS      7175       3084760   
10                        LOV TELECOMUNICACIONES SAS    200738      10576543   

    INGRESO_POR_LINEA  
0              

## Conclusiones

Las 4 consultas SQL confirman los hallazgos obtenidos 
mediante el análisis visual en Python:

1. **Claro** lidera el mercado con más de 42M de líneas 
   en servicio en Q4 2025, seguido de Movistar y Tigo.

2. El **tráfico de datos** ha crecido sostenidamente entre 
   2023 y 2025 para los tres operadores principales, con 
   Claro manteniendo una ventaja consistente.

3. **Antioquia** lidera la cobertura 5G con la mayor cantidad 
   de municipios conectados, mientras que departamentos de 
   frontera no registran ningún municipio con esta tecnología.

4. **Movistar** genera el mayor ingreso por línea entre los 
   tres operadores grandes, confirmando una base de clientes 
   de mayor valor promedio a pesar de tener menos líneas 
   que Claro.

> Las consultas fueron ejecutadas sobre una base de datos 
SQLite en memoria, construida a partir de los datos 
procesados del Boletín Trimestral MinTIC 2023–2025.